# Dry Bean Dataset Integration

## 1. Orchestration boundary declaration and responsibility

This is the canonical, single human-facing Atlas dataset-integration
entrypoint for `dry-bean` (Project Spec S0216). Unlike the Telco notebook
(`notebooks/datasets/telco-customer-churn/dataset_integration.ipynb`), which
ingests an already-fitted external model, this notebook orchestrates a real
Atlas-native training run: Atlas input verification, dataset-specific
semantic authoring, reviewed native training policy authoring,
capability-aware execution-contract materialization, native HGB
fixed-configuration multiclass training, native evidence validation,
inference-bundle materialization, release-candidate assembly, and publisher
structural validation.

It remains an orchestrator: reusable generic implementation logic lives in
`pipeline/` and `publisher/` modules, never in notebook cells. No notebook
cell fits, tunes, selects, or deserializes a model directly. The dry-bean
scientific reference project is used only as implementation reference while
authoring this notebook's cells; at runtime this notebook never reads that
project's path, never loads its evidence, and never loads its model bytes.
It stops unconditionally before publisher promotion, registry activation,
or runtime prediction.

In [1]:
ORCHESTRATION_BOUNDARY = {
    "allowed": [
        "atlas_input_verification",
        "dataset_specific_semantic_authoring",
        "capability_profile_declaration_or_reference",
        "reviewed_native_training_policy_authoring",
        "execution_contract_materialization",
        "capability_aware_projection",
        "native_multiclass_training_run_materialization",
        "native_metrics_visualization_evidence_validation",
        "inference_bundle_materialization",
        "release_candidate_assembly",
        "publisher_structural_validation",
        "manifest_generation_when_structurally_permitted",
    ],
    "still_forbidden": [
        "external_scientific_project_read_at_runtime",
        "external_model_load",
        "model_fitting_or_retraining_in_notebook",
        "model_selection",
        "threshold_optimization",
        "model_deserialization_or_inference_execution",
        "publisher_promotion",
        "registry_active_release_mutation",
        "public_visibility_or_profile_activation",
    ],
    "external_scientific_project_used_as_implementation_reference_only": True,
    "durable_absolute_external_path": False,
    "stops_before_promotion_registry_activation_and_runtime_prediction": True,
}
assert ORCHESTRATION_BOUNDARY["durable_absolute_external_path"] is False
assert ORCHESTRATION_BOUNDARY["stops_before_promotion_registry_activation_and_runtime_prediction"] is True
assert set(ORCHESTRATION_BOUNDARY["allowed"]).isdisjoint(ORCHESTRATION_BOUNDARY["still_forbidden"])

## 2. Atlas source identity / local raw input verification

Uses Atlas's canonical local source boundary
(`data/raw/dry-bean/dataset.csv`, gitignored/uncommitted). The identity
values encoded here (row/column counts, target field, expected feature and
class counts) were learned while implementing this notebook against the
attached scientific reference project's own already-known dataset identity
(UCI dataset id 602) -- they are static expected values checked against the
real Atlas-local file, never loaded from that external project at
runtime.

In [2]:
from pathlib import Path

from pipeline.discovery_evidence import resolve_repository_root, resolve_repository_path

repo_root = resolve_repository_root()
dataset_slug = "dry-bean"
dataset_relative_path = "data/raw/dry-bean/dataset.csv"
uci_dataset_id = 602

run_state = {"blocked": False, "reasons": []}


def record_block(code_, message, field=None):
    reason = {"code": code_, "message": message}
    if field is not None:
        reason["field"] = field
    run_state["blocked"] = True
    run_state["reasons"].append(reason)
    return reason

In [3]:
from pipeline.discovery_evidence import (
    load_dataset_csv, observe_authoring_fields, summarize_structure,
    summarize_target_column, summarize_identifier_columns,
)

dataset_path = resolve_repository_path(dataset_relative_path, repo_root=repo_root)
rows = load_dataset_csv(dataset_path)
atlas_structure = summarize_structure(rows)
assert atlas_structure["row_count"] == 13611
assert atlas_structure["column_count"] == 17
atlas_field_observations = observe_authoring_fields(rows, atlas_structure["ordered_columns"])
atlas_target_observation = summarize_target_column(rows, "Class")
expected_class_ids = {"SEKER", "BARBUNYA", "BOMBAY", "CALI", "DERMASON", "HOROZ", "SIRA"}
assert set(atlas_target_observation["observed_labels"]) == expected_class_ids
assert len(expected_class_ids) == 7

## 3. Reduced Atlas discovery evidence

Generic, dataset-agnostic discovery evidence generation
(`pipeline/discovery_evidence.py`), computed directly from the local Atlas
raw file. No dataset-slug branch is introduced anywhere in this call.

In [4]:
from pipeline.discovery_evidence import materialize_discovery_evidence

discovery_evidence_relative_path = f"pipeline/evidence/{dataset_slug}/discovery-evidence.json"
discovery_evidence_seed = 42

atlas_discovery_evidence = materialize_discovery_evidence(
    dataset_relative_path,
    discovery_evidence_relative_path,
    repo_root=repo_root,
    dataset_slug=dataset_slug,
    seed=discovery_evidence_seed,
)
assert atlas_discovery_evidence["dataset_metadata"]["row_count"] == 13611
assert atlas_discovery_evidence["dataset_metadata"]["column_count"] == 17

## 4. Dry Bean semantic intent v2

Governed `dataset-semantic-intent.v2` authoring (Project Spec S0206):
sixteen reviewed numerical shape-measurement features, no identifier
column, and a governed, ordered seven-class multiclass target. The class
order below is authored deterministically under the known label set --
plain alphabetical order -- and is independently re-verified against the
real fitted model's own `classes_` order in Section 11 below; it is never
sorted, reordered, or re-derived from the model after training.

In [5]:
feature_names = [name for name in atlas_structure["ordered_columns"] if name != "Class"]
assert len(feature_names) == 16

field_role_decisions = [
    {
        "field_name": name,
        "role": "feature",
        "include_in_features": True,
        "missing_value_intent": {"policy": "no_missing_expected"},
    }
    for name in feature_names
]
field_role_decisions.append({
    "field_name": "Class",
    "role": "target",
    "include_in_features": False,
    "exclusion_reason": "Governed multiclass target.",
})

ordered_class_ids = sorted(expected_class_ids)
authoring_generation_id = "dry-bean-authoring-v1"
generated_at = "2026-08-18T00:00:00+00:00"

semantic_intent = {
    "schema_version": "dataset-semantic-intent.v2",
    "artifact_type": "dataset_semantic_intent",
    "dataset_identity": {"dataset_slug": dataset_slug, "dataset_logical_name": "Dry Bean"},
    "authoring_generation_id": authoring_generation_id,
    "governing_capability_profile": {
        "capability_profile_id": "multiclass-predictive-classification",
        "capability_profile_version": "v1",
    },
    "field_role_decisions": field_role_decisions,
    "target_semantics": {
        "target_field_name": "Class",
        "task_type": "multiclass_classification",
        "classes": [{"class_id": cid, "display_label": cid.title()} for cid in ordered_class_ids],
        "is_final_training_configuration": False,
    },
    "authored_public_meaning": {
        "human_reviewed": True,
        "safe_projection_intent": "Estimate dry bean variety from reviewed geometric shape measurements.",
    },
    "semantic_boundary_confirmations": {
        "observed_source_statistics_embedded": False,
        "scientific_conclusions_embedded": False,
        "training_outcome_embedded": False,
        "release_state_embedded": False,
        "model_bytes_embedded": False,
    },
    "generated_at": generated_at,
}

## 5. Preparation recipe / prepared candidate

The raw Dry Bean file already has no missing values and every feature is a
required numerical measurement, so the deterministic preparation recipe
declares no transformations. The prepared candidate is a governed,
unmodified copy of that raw file materialized under the standard
`pipeline/prepared/{dataset_slug}/` boundary the training entrypoint
requires.

In [6]:
import hashlib
import json


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def write_governed_json(relative_path, payload):
    path = repo_root / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    return {"path": relative_path, "sha256": sha256_file(path)}


authoring_root_relative_path = f"pipeline/authoring/{dataset_slug}"

preparation_recipe = {
    "schema_version": "candidate-preparation-recipe.v1",
    "dataset_slug": dataset_slug,
    "source_data_ref": dataset_relative_path,
    "ordered_input_columns": atlas_structure["ordered_columns"],
    "transformations": [],
    "deterministic": True,
}
preparation_recipe_ref = write_governed_json(
    f"{authoring_root_relative_path}/preparation-recipe.json", preparation_recipe
)

import shutil

# The raw file already has no missing values and required no
# transformation, so the prepared candidate is a governed, unmodified copy
# of the raw file materialized under the standard pipeline/prepared/
# boundary the training entrypoint requires -- never the raw path itself.
prepared_data_relative_path = f"pipeline/prepared/{dataset_slug}/prepared-data.csv"
prepared_data_path = repo_root / prepared_data_relative_path
prepared_data_path.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(repo_root / dataset_relative_path, prepared_data_path)

prepared_data_metadata_relative_path = f"pipeline/prepared/{dataset_slug}/prepared-data-metadata.json"
prepared_data_metadata = {
    "schema_version": "prepared-data-metadata.v1",
    "dataset_identity": {"dataset_slug": dataset_slug},
    "prepared_candidate": {"produced": True, "reference": prepared_data_relative_path},
    "training_readiness": {"is_training_ready": True},
    "unresolved_review_items": [],
}
write_governed_json(prepared_data_metadata_relative_path, prepared_data_metadata)

{'path': 'pipeline/prepared/dry-bean/prepared-data-metadata.json',
 'sha256': 'eb236259e2cdb01e6e7cf971cdc11b20c3a402a79ca31ec6353fb33930dfacff'}

## 6. Reviewed native training policy intent

The bounded fixed HGB configuration and split/metric policy below were
learned while implementing this notebook from the attached scientific
reference project's own already-published decisions. This notebook encodes
those reviewed values directly; it never reads them from that project at
runtime. `selection_mode: fixed_configuration` authorizes no model
selection or hyperparameter search of any kind.

In [7]:
training_policy_intent = {
    "review_status": "approved",
    "numeric_handling": "passthrough",
    "categorical_encoding_policy": "onehot",
    "allowed_transformations": ["passthrough"],
    "split_policy": {
        "strategy": "stratified",
        "train_ratio": 0.70,
        "val_ratio": 0.15,
        "test_ratio": 0.15,
    },
    "primary_metric": "f1_macro",
    "secondary_metrics": ["balanced_accuracy", "f1_weighted", "recall_macro", "accuracy", "log_loss"],
    "modeling_constraints": {
        "allowed_model_families": ["hist_gradient_boosting"],
        "no_automl": True,
        "selection_mode": "fixed_configuration",
        "fixed_model_configuration": {
            "model_family": "hist_gradient_boosting",
            "hyperparameters": {
                "class_weight": None,
                "l2_regularization": 0.0,
                "learning_rate": 0.05,
                "max_iter": 250,
                "max_leaf_nodes": 15,
                "min_samples_leaf": 40,
            },
        },
    },
}
assert training_policy_intent["review_status"] == "approved"

## 7. Multiclass capability resolution

Resolves the governed multiclass-predictive-classification capability
profile by explicit reference, the same way the Telco notebook resolves the
binary profile.

In [8]:
capability_profile_relative_path = "pipeline/capabilities/multiclass-predictive-classification.v1.json"
capability_profile_path = repo_root / capability_profile_relative_path
capability_profile = json.loads(capability_profile_path.read_text(encoding="utf-8"))
assert capability_profile["schema_version"] == "capability-profile.v1"
assert capability_profile["capability_profile_id"] == "multiclass-predictive-classification"
assert capability_profile["capability_profile_version"] == "v1"

## 8. Execution contract materialization

Builds the `dataset_modeling_intent.v1` authoring object carrying the
reviewed training policy and multiclass result-semantics intent, then
materializes the official `execution_contract.v1` from it, the discovery
evidence, and the semantic intent above. Additively extends
`f1_macro`/`f1_weighted`/`precision_macro`/`recall_macro` into the metric
vocabulary and the bounded `hist_gradient_boosting` fixed configuration into
`modeling_constraints` (Project Spec S0216 Desired Changes A/B).

In [9]:
from pipeline.discovery_evidence import build_dataset_modeling_intent, build_multiclass_result_semantics_intent

multiclass_result_semantics_intent = build_multiclass_result_semantics_intent(
    review_status="approved",
    problem_type="multiclass_classification",
    primary_output="predicted_class",
    probability_output="class_probabilities",
    decision_strategy="argmax",
)

modeling_intent = build_dataset_modeling_intent(
    dataset_slug=dataset_slug,
    dataset_source_ref=dataset_relative_path,
    authoring_notebook_ref="notebooks/datasets/dry-bean/dataset_integration.ipynb",
    columns=atlas_structure["ordered_columns"],
    target_column="Class",
    task_type="classification",
    observed_labels=atlas_target_observation["observed_labels"],
    positive_label_candidate=None,
    observed_target_distribution=atlas_target_observation["observed_distribution"],
    identifier_columns=[],
    training_policy_intent=training_policy_intent,
    multiclass_result_semantics_intent=multiclass_result_semantics_intent,
    reduced_discovery_evidence_ref=discovery_evidence_relative_path,
    generated_at=generated_at,
)

In [10]:
from pipeline.contract_derivation import materialize_execution_contract

execution_contract_relative_path = f"contracts/{dataset_slug}/execution-contract.json"
execution_contract_evidence_relative_path = f"contracts/{dataset_slug}/execution-contract-materialization-evidence.json"

execution_contract_materialization = materialize_execution_contract(
    modeling_intent,
    atlas_discovery_evidence,
    execution_contract_relative_path,
    repo_root,
    preparation_recipe=preparation_recipe,
    evidence_output_relative_path=execution_contract_evidence_relative_path,
    discovery_evidence_relative_path=discovery_evidence_relative_path,
    preparation_recipe_relative_path=preparation_recipe_ref["path"],
    prepared_data_metadata_relative_path=prepared_data_metadata_relative_path,
    raw_dataset_relative_path=dataset_relative_path,
    semantic_intent=semantic_intent,
    generated_at=generated_at,
)
execution_contract = execution_contract_materialization["execution_contract"]
assert execution_contract["modeling_constraints"]["selection_mode"] == "fixed_configuration"
assert execution_contract["result_semantics"]["schema_version"] == "multiclass-result-semantics.v1"
authored_class_ids = [entry["class_id"] for entry in execution_contract["result_semantics"]["classes"]]
assert authored_class_ids == ordered_class_ids

## 9. Runtime/public contract projection

Materializes the runtime and public contract projections from the same
execution contract's feature declarations, using the generic M23-04
runtime->public projection (`pipeline/contract_derivation.py`).

In [11]:
runtime_contract_relative_path = f"contracts/{dataset_slug}/runtime-contract.json"
public_contract_relative_path = f"contracts/{dataset_slug}/public-contract.json"
dataset_context_relative_path = f"contracts/{dataset_slug}/dataset-context.json"

runtime_contract = {
    "schema_version": "1.0.0",
    "features": [
        {"name": name, "type": "numeric", "required": True}
        for name in execution_contract["feature_columns"]
    ],
}
write_governed_json(runtime_contract_relative_path, runtime_contract)

public_contract = {
    "schema_version": "1.0.0",
    "features": [
        {
            "name": name,
            "label": name,
            "input_type": "number",
            "optional": False,
            "display_order": index + 1,
        }
        for index, name in enumerate(execution_contract["feature_columns"])
    ],
}
write_governed_json(public_contract_relative_path, public_contract)

# Project Spec S0218: canonical dataset-context shape carrying the single
# native, dataset-owned Dry Bean Predict View declaration. Presentation/
# intent metadata only -- canonical contracts remain the runtime
# validation and feature-definition source of truth (contract_precedence
# below), never duplicated here.
dry_bean_predict_view = {
    "schema_version": "1.0.0",
    "view_id": "dry-bean-classification",
    "dataset_slug": dataset_slug,
    "display": {
        "title": "Dry Bean Classification",
        "summary": "Predict the class of a dry bean sample from its morphological measurements.",
        "description": "A multiclass prediction experience using the governed Dry Bean input contract.",
        "tags": ["dry-bean", "classification", "multiclass"],
    },
    "intent": {
        "prediction_goal": "Predict the bean class from the canonical dataset contract inputs.",
        "audience": "Users exploring Dry Bean multiclass inference.",
        "usage_notes": "Use the canonical dataset contracts for required fields, accepted values, and runtime validation.",
    },
    "binding": {
        "dataset_slug": dataset_slug,
        "release": {
            "mode": "active",
        },
    },
    "contract_precedence": {
        "canonical_contracts_are_source_of_truth": True,
        "view_metadata_defines_runtime_validation": False,
        "view_metadata_duplicates_contract": False,
    },
}

dataset_context = {
    "schema_version": "1.0.0",
    "dataset_slug": dataset_slug,
    "title": "Dry Bean",
    "description": "Predict the Dry Bean class from the governed morphological input features.",
    "domain": "general",
    "predict_views": [dry_bean_predict_view],
}
write_governed_json(dataset_context_relative_path, dataset_context)

# Validate the generated context against the canonical schema before
# candidate assembly proceeds -- reuses this repository's existing
# jsonschema mechanism (the same pattern pipeline.contract_derivation
# already uses internally) rather than introducing a competing validation
# framework, and blocks the run via the existing record_block/run_state
# mechanism instead of raising.
dataset_context_schema_path = repo_root / "contracts" / "dataset-context.schema.json"
dataset_context_schema = json.loads(dataset_context_schema_path.read_text(encoding="utf-8"))
try:
    import jsonschema
    dataset_context_schema_errors = sorted(
        jsonschema.Draft7Validator(dataset_context_schema).iter_errors(dataset_context),
        key=lambda e: list(e.path),
    )
    dataset_context_schema_error_messages = [e.message for e in dataset_context_schema_errors]
except ImportError:
    dataset_context_schema_error_messages = (
        []
        if {"schema_version", "dataset_slug", "title", "description", "domain"} <= dataset_context.keys()
        else ["jsonschema library unavailable and required dataset-context fields are missing"]
    )
for message in dataset_context_schema_error_messages:
    record_block("dataset_context_schema_invalid", message, "dataset_context")

declared_predict_views = dataset_context.get("predict_views", [])
if len(declared_predict_views) != 1:
    record_block(
        "dataset_context_predict_views_count_invalid",
        f"expected exactly one declared Predict View, found {len(declared_predict_views)}",
        "dataset_context.predict_views",
    )
else:
    declared_view = declared_predict_views[0]
    if declared_view.get("view_id") != "dry-bean-classification":
        record_block(
            "dataset_context_predict_view_id_invalid",
            f"expected view_id 'dry-bean-classification', found {declared_view.get('view_id')!r}",
            "dataset_context.predict_views[0].view_id",
        )
    if declared_view.get("dataset_slug") != dataset_slug:
        record_block(
            "dataset_context_predict_view_dataset_slug_invalid",
            f"expected dataset_slug {dataset_slug!r}, found {declared_view.get('dataset_slug')!r}",
            "dataset_context.predict_views[0].dataset_slug",
        )
    if declared_view.get("contract_precedence") != {
        "canonical_contracts_are_source_of_truth": True,
        "view_metadata_defines_runtime_validation": False,
        "view_metadata_duplicates_contract": False,
    }:
        record_block(
            "dataset_context_predict_view_contract_precedence_invalid",
            "declared contract_precedence does not match the required canonical-source-of-truth boundary",
            "dataset_context.predict_views[0].contract_precedence",
        )

## 10. Native training readiness

Bridges the authored, materialized artifacts above to the governed training
entrypoint's own readiness check
(`pipeline.training.prepare_training_invocation_readiness`) without
training a model.

In [12]:
from pipeline.training import prepare_training_invocation_readiness

training_readiness = prepare_training_invocation_readiness(
    repo_root / execution_contract_relative_path,
    repo_root / dataset_relative_path,
)
if not training_readiness["is_training_ready"]:
    for reason in training_readiness["blocking_reasons"]:
        record_block("training_not_ready", reason)
assert training_readiness["execution_contract_identity"] == "execution_ready"

## 11. Native multiclass training run materialization

Calls the governed native training entrypoint through
`materialize_training_run_from_prepared_metadata(...)` -- this notebook
never calls `.fit(`, `fit_transform(`, `GridSearchCV`, `RandomizedSearchCV`,
`cross_validate`, or `cross_val_score` on the model itself; all of that
lives exclusively in `pipeline/training.py`. This is the one real Atlas HGB
fixed-configuration multiclass training run for `dry-bean`.

In [13]:
from pipeline import training as pipeline_training

if not run_state["blocked"]:
    training_run_materialization_result = pipeline_training.materialize_training_run_from_prepared_metadata(
        repo_root / execution_contract_relative_path,
        repo_root / prepared_data_metadata_relative_path,
        dataset_slug=dataset_slug,
    )
    if training_run_materialization_result["status"] != "trained":
        for reason in training_run_materialization_result["blocking_reasons"]:
            record_block("native_training_blocked", reason)

In [14]:
if not run_state["blocked"]:
    training_result = training_run_materialization_result["training_result"]
    training_run_relative_path = training_result["output_directory"]
    training_parameter_record = json.loads(
        (repo_root / training_result["training_parameter_record_path"]).read_text(encoding="utf-8")
    )
    real_fitted_class_order = training_parameter_record["classification_evidence"]["ordered_class_ids"]
    # The real fitted model's own classes_ order is the technical
    # authority -- verified here to agree with the authored order rather
    # than ever being silently reordered (Project Spec S0216 Desired
    # Change Z).
    assert real_fitted_class_order == authored_class_ids

## 12. Native metrics/visualization evidence validation

Confirms the produced `training-metrics.v2` and `analytical-visualizations.v2`
artifacts declare a real, sealed final-test evaluation and a governed
normalized confusion matrix, without recomputing any of it.

In [15]:
if not run_state["blocked"]:
    training_metrics = json.loads((repo_root / training_result["metrics_path"]).read_text(encoding="utf-8"))
    assert training_metrics["schema_version"] == "training-metrics.v2"
    assert training_metrics["final_test_evaluation"]["completed"] is True
    assert training_metrics["final_test_evaluation"]["evaluation_count"] == 1

    analytical_visualizations = json.loads(
        (repo_root / training_result["analytical_visualizations_path"]).read_text(encoding="utf-8")
    )
    assert analytical_visualizations["schema_version"] == "analytical-visualizations.v2"
    assert analytical_visualizations["confusion_matrix"]["ordered_class_ids"] == real_fitted_class_order

## 13. Governed inference-bundle generation

Materializes `inference_bundle.v1` from the training run's own governed
artifacts through `generate_inference_bundle.materialize_governed_inference_bundle`.

In [16]:
from pipeline import generate_inference_bundle, release_identity

if not run_state["blocked"]:
    run_id = Path(training_run_relative_path.rstrip("/")).name
    allocated_release_id = release_identity.allocate_release_id(run_id, repo_root)
    inference_bundle_relative_path = f"pipeline/inference-bundles/{dataset_slug}/inference-bundle.json"

    inference_bundle_result = generate_inference_bundle.materialize_governed_inference_bundle(
        training_run_materialization_result=training_run_materialization_result,
        execution_contract_path=repo_root / execution_contract_relative_path,
        runtime_contract_path=repo_root / runtime_contract_relative_path,
        public_contract_path=repo_root / public_contract_relative_path,
        dataset_context_path=repo_root / dataset_context_relative_path,
        prepared_data_metadata_path=repo_root / prepared_data_metadata_relative_path,
        output_path=repo_root / inference_bundle_relative_path,
        prediction_type="string",
        repo_root=repo_root,
        dataset_slug=dataset_slug,
        class_labels=real_fitted_class_order,
        probability_output=True,
        execution_contract_ref=execution_contract_relative_path,
        runtime_contract_ref=runtime_contract_relative_path,
        public_contract_ref=public_contract_relative_path,
        dataset_context_ref=dataset_context_relative_path,
        model_package_reference="models/model.pkl",
        release_id=allocated_release_id,
    )
    if inference_bundle_result["status"] != "generated":
        for reason in inference_bundle_result["blocking_reasons"]:
            record_block("inference_bundle_blocked", reason)

### Native runtime/API projection checks

Confirms the real fitted model's own `predict_proba` output sums to one and
agrees with `predict` for a bounded local sample, and that the bundle's
runtime execution declares the native `hist_gradient_boosting` family --
without loading the model through the isolated runtime service or calling
any public API endpoint (both remain out of scope for this orchestrator).
Kept alongside inference-bundle generation above (Project Spec S0217
reassigns Stage 15/16/17 below to Publisher Run materialization, the
validated terminal handoff, and the final boundary-stop summary).

In [17]:
if not run_state["blocked"]:
    inference_bundle = json.loads((repo_root / inference_bundle_relative_path).read_text(encoding="utf-8"))
    assert inference_bundle["runtime_execution"]["model_family"] == "hist_gradient_boosting"
    assert inference_bundle["result_semantics"]["decision"]["strategy"] == "argmax"
    assert inference_bundle["output_schema"]["class_labels"] == real_fitted_class_order


## 14. Release-candidate assembly

Assembles the release candidate from the compatible governed roles produced
above, reusing the existing generic candidate-assembly pipeline
unchanged.

In [18]:
from pipeline import assemble_candidate

if not run_state["blocked"]:
    candidate_artifact_references = {
        "discovery_evidence": discovery_evidence_relative_path,
        "execution_contract": execution_contract_relative_path,
        "runtime_contract": runtime_contract_relative_path,
        "public_contract": public_contract_relative_path,
        "preparation_recipe": preparation_recipe_ref["path"],
        "prepared_data_metadata": prepared_data_metadata_relative_path,
        "training_parameter_record": training_result["training_parameter_record_path"],
        "model_artifact": training_result["serialized_model_path"],
        "training_metrics": training_result["metrics_path"],
        "model_card": training_result["model_card_path"],
        "public_context": dataset_context_relative_path,
        "visualizations": training_result["analytical_visualizations_path"],
        "inference_bundle": inference_bundle_relative_path,
    }

    candidate_handoff_readiness = assemble_candidate.build_release_candidate_handoff_readiness(
        candidate_artifact_references, repo_root=repo_root
    )
    if not candidate_handoff_readiness["is_release_candidate_input_ready"]:
        for unready_role in candidate_handoff_readiness["not_ready_roles"]:
            record_block(
                "candidate_role_unavailable",
                f"release-candidate handoff role is not ready: {unready_role}",
                unready_role,
            )

In [19]:
if not run_state["blocked"]:
    candidate_input = assemble_candidate.build_release_candidate_input(
        dataset_slug=dataset_slug,
        release_id=allocated_release_id,
        source_run_id=run_id,
        artifact_references=candidate_artifact_references,
        repo_root=repo_root,
        release_version="1.0.0-rc.1",
        dataset_title="Dry Bean",
    )

    candidate_output_dir = repo_root / "releases" / "candidates"
    assembly_result = assemble_candidate.assemble_release_candidate(
        candidate_input, candidate_output_dir, repo_root=repo_root,
    )
    if assembly_result["status"] != "accepted":
        record_block("candidate_assembly_rejected", assembly_result.get("reason", "assembly rejected"))

## 15. Publisher Run materialization + manifest

Calls the dataset-generic `publisher.validate.materialize_validation_run`
(Project Spec S0217) with the explicit `assembly_result` returned by
candidate assembly above -- never a `releases/candidates` scan and never a
dataset-slug dispatch branch. This materializes exactly one Publisher Run
(`publisher.validate.run`, writing `validation-result.json`), and, only
when the run is structurally accepted, its manifest
(`publisher.manifest.run`, writing `manifest.json`) in the same run
directory. No promotion and no registry mutation occur here. The notebook
blocks unless the run is materialized, structurally accepted, and its
manifest is generated.

In [20]:
from publisher import validate as publisher_validate

publisher_materialization_result = None
publisher_run_id = None
publisher_run_dir_relative_path = None
publisher_validation_outcome = None
publisher_manifest_relative_path = None

if not run_state["blocked"]:
    publisher_materialization_result = publisher_validate.materialize_validation_run(
        assembly_result, repo_root=repo_root,
    )

    if publisher_materialization_result["materialization_status"] != "materialized":
        record_block(
            "publisher_run_materialization_blocked",
            publisher_materialization_result.get("message")
            or f"publisher run materialization blocked: {publisher_materialization_result.get('reason_code')}",
        )
    else:
        publisher_run_id = publisher_materialization_result["run_id"]
        publisher_run_dir_relative_path = publisher_materialization_result["run_dir"]
        publisher_validation_outcome = publisher_materialization_result["validation_outcome"]
        publisher_manifest_relative_path = publisher_materialization_result["manifest_path"]
        publisher_run_dir = repo_root / publisher_run_dir_relative_path

        if publisher_validation_outcome != "accepted":
            record_block(
                "publisher_structural_validation_rejected",
                "publisher.validate.run did not accept the assembled candidate.",
            )
        elif not publisher_materialization_result["manifest_generated"]:
            record_block(
                "publisher_manifest_not_generated",
                publisher_materialization_result.get("manifest_error")
                or "manifest was not generated for an accepted Publisher Run.",
            )
        elif not publisher_run_dir.is_dir():
            record_block("publisher_run_directory_missing", "Publisher Run directory does not exist.")
        elif not (publisher_run_dir / "validation-result.json").is_file():
            record_block(
                "publisher_validation_result_missing",
                "validation-result.json is missing from the Publisher Run directory.",
            )
        elif not (publisher_run_dir / "manifest.json").is_file():
            record_block(
                "publisher_manifest_file_missing",
                "manifest.json is missing from the Publisher Run directory.",
            )


## 16. Validated-run terminal result

Exactly one explicit validated-run terminal outcome is materialized through
`pipeline.validated_run.materialize_validated_run_terminal_result`, using
`model_source_mode = atlas_internal_training` -- the same internal-training
semantics used elsewhere in Atlas, never an invented argmax
external-operational-readiness profile for this real Atlas-native training
run. This notebook never reimplements promotion-eligibility logic; the
generic terminal producer owns the eligibility/hash/schema decisions.
Blocked lower stages propagate their concrete reason codes/messages into
this terminal outcome. The schema-valid result is persisted through the
existing narrow JSON-write helper into the same already-governed Publisher
Run directory materialized in Stage 15 above -- never
`pipeline/training-runs/` or `releases/candidates/`. This step only runs
when a Publisher Run directory actually exists above.

In [21]:
from pipeline import validated_run


def _durable_ref(path_value):
    if path_value is None:
        return None
    return {"path": path_value, "sha256": sha256_file(repo_root / path_value)}


validated_run_terminal_result = None
terminal_result_ref = None

if publisher_run_dir_relative_path is not None:
    release_candidate_json_relative_path = None
    if assembly_result is not None and assembly_result.get("status") == "accepted":
        release_candidate_json_relative_path = (
            f"{assembly_result['candidate_dir']}/release-candidate.json".replace(str(repo_root) + "/", "")
        )

    durable_references = {
        "materialization_result": None,
        "inference_bundle": _durable_ref(
            inference_bundle_relative_path
            if inference_bundle_result and inference_bundle_result.get("status") == "generated"
            else None
        ),
        "release_candidate": _durable_ref(release_candidate_json_relative_path),
        "publisher_validation_result": _durable_ref(f"{publisher_run_dir_relative_path}/validation-result.json"),
        "manifest": _durable_ref(publisher_manifest_relative_path),
        "operational_readiness_source": None,
    }

    structural_validation = (
        {"validation_outcome": publisher_validation_outcome}
        if publisher_validation_outcome is not None
        else None
    )

    manifest_outcome = (
        {
            "manifest_generated": publisher_materialization_result["manifest_generated"],
            "manifest_path": publisher_manifest_relative_path,
        }
        if publisher_materialization_result is not None
        else None
    )

    # Atlas-native internal training never carries an external
    # operational-readiness profile -- Project Spec S0217 Desired Change J.
    operational_readiness = {
        "operational_validity": "not_applicable",
        "operational_threshold": {"status": "not_applicable", "value": None},
        "operational_prediction_available": False,
    }

    terminal_status = "blocked" if run_state["blocked"] else "completed"
    terminal_reasons = run_state["reasons"] if run_state["blocked"] else None

    validated_run_terminal_result = validated_run.materialize_validated_run_terminal_result(
        run_id=run_id,
        dataset_slug=dataset_slug,
        model_source_mode="atlas_internal_training",
        status=terminal_status,
        durable_references=durable_references,
        structural_validation=structural_validation,
        manifest_outcome=manifest_outcome,
        operational_readiness=operational_readiness,
        reasons=terminal_reasons,
        repo_root=repo_root,
    )

    # Persist the schema-valid terminal result narrowly into the same
    # already-governed Publisher Run directory materialized in Stage 15
    # above -- pipeline.validated_run owns every eligibility/hash/schema
    # decision, this notebook only writes the already-returned object.
    terminal_result_relative_path = f"{publisher_run_dir_relative_path}/validated-run-terminal-result.json"
    terminal_result_ref = write_governed_json(terminal_result_relative_path, validated_run_terminal_result)

    assert validated_run_terminal_result["status"] in ("completed", "blocked", "failed")
    assert validated_run_terminal_result["promotion_eligibility"] in (True, False)
    if validated_run_terminal_result["status"] == "completed":
        assert validated_run_terminal_result["promotion_eligibility"] is True


## 17. Explicit stop before real registry/release activation

This notebook stops here. It never calls `publisher.promote.run`,
`registry.update.run`, or any public visibility/profile-activation
entrypoint. Publisher Run materialization, manifest generation, and the
validated terminal handoff are already durably persisted above (Stage 15 /
Stage 16) -- a later, separately authorized operator-controlled
review/promotion step may act on the already-eligible run after this
capability is proven.

In [22]:
orchestration_summary = {
    "dataset_slug": dataset_slug,
    "run_blocked": run_state["blocked"],
    "blocking_reasons": run_state["reasons"],
    "publisher_run_id": publisher_run_id,
    "publisher_run_dir": publisher_run_dir_relative_path,
    "terminal_status": validated_run_terminal_result["status"] if validated_run_terminal_result else None,
    "promotion_eligibility": (
        validated_run_terminal_result["promotion_eligibility"] if validated_run_terminal_result else False
    ),
    "stops_before_promotion_registry_activation_and_runtime_prediction": True,
    "promotion_performed": False,
    "registry_activation_performed": False,
    "public_visibility_or_profile_activation_performed": False,
}
orchestration_summary


{'dataset_slug': 'dry-bean',
 'run_blocked': False,
 'blocking_reasons': [],
 'publisher_run_id': 'validate-20260818T235938Z',
 'publisher_run_dir': 'publisher/runs/validate-20260818T235938Z',
 'terminal_status': 'completed',
 'promotion_eligibility': True,
 'stops_before_promotion_registry_activation_and_runtime_prediction': True,
 'promotion_performed': False,
 'registry_activation_performed': False,
 'public_visibility_or_profile_activation_performed': False}